In [4]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

#Load environment variables 
from helper import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Set OpenAI model

In [ ]:
os.environ['OPENAI_MODEL_NAME'] = "gpt-4o-mini"

print(os.getenv("OPENAI_API_KEY"))

### Load the task and agent ymal file

In [7]:
#Define the file path for yaml configuration
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

#Load configuration from yaml file
configs = {}

for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

#Assign loaded configuratio to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

### Create Pydantic models for structured output

In [8]:
from typing import List
from pydantic import BaseModel, Field

class TaskEstimate(BaseModel):
    task_name: str = Field(..., description="Name of the task")
    estimated_time_hours: float = Field(..., description="Estimated time to complete the task in hours")
    required_resources: List[str] = Field(..., description="List of resources required to complete the task")

class Milestone(BaseModel):
    milestone_name: str = Field(..., description="Name of the milestone")
    tasks: List[str] = Field(..., description="List of the task IDs associated with this milestone")

class ProjectPlan(BaseModel):
    tasks: List[TaskEstimate] = Field(..., description="List of task with their estimates")
    milestones: List[Milestone] = Field(..., description="List of project milestones") 


### Create Crew, Agents and Tasks

In [18]:
#Creating Agents
project_planning_agent = Agent(
    config=agents_config['project_planning_agent']
)

estimation_agent = Agent(
    config = agents_config['estimation_agent']
)

resource_allocation_agent = Agent(
    config = agents_config['resource_allocation_agent']
)

#Creating task
task_breakdown = Task(
    config=tasks_config['task_breakdown'],
    agent = project_planning_agent
)

time_resource_estimation = Task(
    config= tasks_config['time_resource_estimation'],
    agent=estimation_agent
)

resource_allocation = Task(
    config=tasks_config['resource_allocation'],
    agent=resource_allocation_agent,
    output_pydantic=ProjectPlan # This is the structured output we want
)

#Creating crew
crew = Crew(
    agents=[
        project_planning_agent,
        estimation_agent, 
        resource_allocation_agent
    ],
    tasks=[
        task_breakdown,
        time_resource_estimation,
        resource_allocation
    ],
    verbose=True
)

In [19]:
print(crew.tasks)

[Task(description=Carefully analyze the project_requirements for the {project_type} project and break them down into individual tasks. Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are accounted for:
{project_requirements}

Team members:
{team_members}
, expected_output=A comprehensive list of tasks with detailed descriptions, timelines, dependencies, and deliverables. Your final output MUST include a Gantt chart or similar timeline visualization specific to the {project_type} project.
), Task(description=Thoroughly evaluate each task in the {project_type} project to estimate the time, resources, and effort required. Use historical data, task complexity, and available resources to provide a realistic estimation for each task.
, expected_output=A detailed estimation report outlining the time, resources, and effort required for each task in the {project_type} project. Your final report MUST include a summary of any risks or uncertainties a

### Crew's Inputs 


In [11]:
from IPython.display import display, Markdown

project = "Website"
industry = 'Technology'
project_objectives = 'Create a webiste for a small business'
team_members = """
- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)
"""
project_requirements = """
- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust
"""

#formate the disctionary as markdown for a better display in jupyter lab
formatted_output = f"""
**Project Type:** {project}

**Project Objectives:** {project_objectives}

**Industry:** {industry}

**Team Members:**
{team_members}

**Project Requirements:**
{project_requirements}

"""

#Display the formatted output as markdown
display(Markdown(formatted_output))


**Project Type:** Website

**Project Objectives:** Create a webiste for a small business

**Industry:** Technology

**Team Members:**

- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)


**Project Requirements:**

- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust




### Kicking off the crew

In [20]:
#The given pythong dictionary
inputs = {
    'project_type': project,
    'project_objectives': project_objectives,
    'industry': industry,
    'team_members': team_members,
    'project_requirements': project_requirements
}

#Run the crew
result = crew.kickoff(
    inputs=inputs
)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 57553806-693a-4580-8eab-c206a032b676                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Ultimate Project Planner                                                                            │
│                                                                                                                 │
│  Task: Carefully analyze the project_requirements for the Website project and break them down into individual   │
│  tasks. Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are      │
│  accounted for:                                                                                                 │
│                                                                                                                 │
│  - Create a responsive design that works well on desktop and mobile devices                                     │
│  - Implement a modern, visually appealing user interface with a clean look                                      │
│  - Develop a user-friendly navigation system with intuitive menu structure                                      │
│  - Include an "About Us" page highlighting the company's history and values                                     │
│  - Design a "Services" page showcasing the business's offerings with descriptions                               │
│  - Create a "Contact Us" page with a form and integrated map for communication                                  │
│  - Implement a blog section for sharing industry news and company updates                                       │
│  - Ensure fast loading times and optimize for search engines (SEO)                                              │
│  - Integrate social media links and sharing capabilities                                                        │
│  - Include a testimonials section to showcase customer feedback and build trust                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Team members:                                                                                                  │
│                                                                                                                 │
│  - John Doe (Project Manager)                                                                                   │
│  - Jane Doe (Software Engineer)                                                                                 │
│  - Bob Smith (Designer)                                                                                         │
│  - Alice Johnson (QA Engineer)                                                                                  │
│  - Tom Brown (QA Engineer)                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Ultimate Project Planner                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Website Project Requirements Breakdown                                                                     │
│                                                                                                                 │
│  **Project Overview**: This project focuses on creating a responsive website for a small business that          │
│  highlights services, company values, and enables user interaction.                                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  #### Task Breakdown                                                                                            │
│                                                                                                                 │
│  **1. Project Initialization**                                                                                  │
│     - **Description**: Kick-off meeting to align on project scope, goals, and deliverables.                     │
│     - **Assigned To**: John Doe                                                                                 │
│     - **Timeline**: Day 1                                                                                       │
│     - **Dependencies**: None                                                                                    │
│     - **Deliverable**: Project Charter Document                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **2. Conduct User Research**                                                                                   │
│     - **Description**: Gather insights on user needs and preferences through surveys and interviews.            │
│     - **Assigned To**: John Doe                                                                                 │
│     - **Timeline**: Day 2-5                                                                                     │
│     - **Dependencies**: Task 1                                                                                  │
│     - **Deliverable**: User Research Report                                                                     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **3. Create Wireframes**                                                                                       │
│     - **Description**: Develop basic layouts for each page, focusing on structure and placement of elements.    │
│     - **Assigned To**: Bob Smith                       

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 9fa11812-d4d0-4cea-b9bb-6519a8cad52b                                                                     │
│  Agent: The Ultimate Project Planner                                                                            │
│                                                                                                                 │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Expert Estimation Analyst                                                                               │
│                                                                                                                 │
│  Task: Thoroughly evaluate each task in the Website project to estimate the time, resources, and effort         │
│  required. Use historical data, task complexity, and available resources to provide a realistic estimation for  │
│  each task.                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 0fb449d1-df61-4ed7-8412-d45bcc60fa18                                                                     │
│  Agent: Resource Allocation Strategist                                                                          │
│                                                                                                                 │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 57553806-693a-4580-8eab-c206a032b676                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {                                                                                                │
│    "tasks": [                                                                                                   │
│      {                                                                                                          │
│        "task_name": "Project Initialization",                                                                   │
│        "estimated_time_hours": 8,                                                                               │
│        "required_resources": ["John Doe"]                                                                       │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Conduct User Research",                                                                    │
│        "estimated_time_hours": 32,                                                                              │
│        "required_resources": ["John Doe"]                                                                       │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Create Wireframes",                                                                        │
│        "estimated_time_hours": 40,                                                                              │
│        "required_resources": ["Bob Smith"]                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Develop Responsive Design",                                                                │
│        "estimated_time_hours": 80,                                                                              │
│        "required_resources": ["Bob Smith"]                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Build User Interface (UI)",                                                                │
│        "estimated_time_hours": 80,                                                                              │
│        "required_resources": ["Jane Doe"]                                                                       │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Develop Navigation System",      

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────────── Execution Traces ──────────────────────────────╮
│                                                                              │
│  🔍 Detailed execution traces are available!                                 │
│                                                                              │
│  View insights including:                                                    │
│    • Agent decision-making process                                           │
│    • Task execution flow and timing                                          │
│    • Tool usage details                                                      │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
Would you like to view your execution traces? [y/N] (20s timeout): 



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

### Usage Metrics and costs

In [ ]:
#let's see how much it would cost each time if this crew runs at scale

import pandas as pd



[]
